# Modular WSB Demo

This notebook shows the baseline modular workflow for `wind + solar + battery` and compares the modular outputs against the original HyDesign implementation for the France Design 1 case.

Use the `hydesign` conda environment as the notebook kernel.

In [ ]:
from __future__ import annotations

import importlib
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from hydesign.assembly.hpp_assembly import hpp_model as original_hpp_model
from hydesign.examples import examples_filepath
from hydesign.modular import build_project
from hydesign.modular.modules.assembly.artifact_codegen import OUTPUT_LABELS
from hydesign.tests.test_files import tfp

In [ ]:
PROJECT_NAME = "notebook_wind_solar_battery_cplex"
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "hydesign" / "__init__.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate the repository root containing hydesign/__init__.py")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

OUTPUT_ROOT = (REPO_ROOT / "hydesign" / "modular" / "projects").resolve()
OUTPUT_DIR = OUTPUT_ROOT / PROJECT_NAME
RTOL = 1e-7
ATOL = 1e-7


def drop_generated_modules(project_name: str) -> None:
    prefix = f"hydesign.modular.projects.{project_name}"
    for module_name in list(sys.modules):
        if module_name == prefix or module_name.startswith(prefix + "."):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()


def load_generated_assembly(project_name: str):
    drop_generated_modules(project_name)
    return importlib.import_module(f"hydesign.modular.projects.{project_name}.assembly")


def baseline_design_1_inputs() -> dict[str, object]:
    examples_sites = pd.read_csv(
        f"{examples_filepath}examples_sites.csv", index_col=0, sep=";"
    )
    ex_site = examples_sites.loc[examples_sites.name == "France_good_wind"]
    output_df = pd.read_csv(tfp + "France_good_wind_design.csv", index_col=0, sep=";")
    design = output_df["Design 1"]
    x = [
        design["clearance [m]"],
        design["sp [W/m2]"],
        design["p_rated [MW]"],
        design["Nwt"],
        design["wind_MW_per_km2 [MW/km2]"],
        design["solar_MW [MW]"],
        design["surface_tilt [deg]"],
        design["surface_azimuth [deg]"],
        design["DC_AC_ratio"],
        design["b_P [MW]"],
        design["b_E_h [h]"],
        design["cost_of_battery_P_fluct_in_peak_price_ratio"],
    ]
    return {
        "latitude": ex_site["latitude"].values[0],
        "longitude": ex_site["longitude"].values[0],
        "altitude": ex_site["altitude"].values[0],
        "sim_pars_fn": examples_filepath + ex_site["sim_pars_fn"].values[0],
        "input_ts_fn": examples_filepath + ex_site["input_ts_fn"].values[0],
        "x": x,
    }


shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR

## 1. Build the modular project

In [ ]:
build_project(
    {
        "project_name": PROJECT_NAME,
        "technologies": ["wind", "solar", "battery"],
        "artifacts": ["assembly"],
        "output_root": str(OUTPUT_ROOT),
        "regenerate": True,
    }
)

## 2. Run the original and modular assemblies on the same case

In [ ]:
inputs = baseline_design_1_inputs()
generated = load_generated_assembly(PROJECT_NAME)

original_work_dir = str((OUTPUT_DIR / "original").resolve()) + os.sep
modular_work_dir = str((OUTPUT_DIR / "modular").resolve()) + os.sep

original_hpp = original_hpp_model(
    latitude=inputs["latitude"],
    longitude=inputs["longitude"],
    altitude=inputs["altitude"],
    max_num_batteries_allowed=10,
    work_dir=original_work_dir,
    sim_pars_fn=inputs["sim_pars_fn"],
    input_ts_fn=inputs["input_ts_fn"],
)
modular_hpp = generated.hpp_model(
    latitude=inputs["latitude"],
    longitude=inputs["longitude"],
    altitude=inputs["altitude"],
    max_num_batteries_allowed=10,
    work_dir=modular_work_dir,
    sim_pars_fn=inputs["sim_pars_fn"],
    input_ts_fn=inputs["input_ts_fn"],
)

original_outputs = np.asarray(original_hpp.evaluate(*inputs["x"]), dtype=float)
modular_outputs = np.asarray(modular_hpp.evaluate(*inputs["x"]), dtype=float)

comparison = pd.DataFrame(
    {
        "metric": OUTPUT_LABELS[: len(original_outputs)],
        "original": original_outputs,
        "modular": modular_outputs,
    }
)
comparison["abs_diff"] = np.abs(comparison["modular"] - comparison["original"])
comparison["rel_diff"] = comparison["abs_diff"] / np.maximum(
    np.abs(comparison["original"]), 1e-12
)

comparison

## 3. Check parity with tolerance

In [ ]:
summary = pd.Series(
    {
        "allclose": np.allclose(modular_outputs, original_outputs, rtol=RTOL, atol=ATOL),
        "max_abs_diff": comparison["abs_diff"].max(),
        "max_rel_diff": comparison["rel_diff"].max(),
        "rtol": RTOL,
        "atol": ATOL,
    }
)
summary

If `allclose` is `True`, the modular WSB assembly matches the original implementation for this reference case within the selected tolerance.